# AI Productivity Maximizer - Regression Model (Phase 2)

This notebook builds **3 regression models** for intelligent task scheduling:
1. **Productivity Score Predictor** - Continuous 0-100 score
2. **Required Study Hours Predictor** - Optimal daily study time
3. **Break Interval Optimizer** - Personalized break durations

These models replace the binary classification approach and feed into AI-based timetable generation.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

import warnings
import joblib
import os
import json

warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print("Ready to build regression-based productivity models.")

## 2. Load & Prepare Dataset

In [ ]:
# Load dataset
df = pd.read_csv('student_habits_performance.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## 3. Data Cleaning

In [ ]:
# Check missing values
print("Missing values before cleaning:")
print(df.isnull().sum())

# Drop ID column
if 'student_id' in df.columns:
    df.drop(columns=['student_id'], inplace=True)

# Fill missing values with mode
df.fillna(df.mode().iloc[0], inplace=True)

print("\nMissing values after cleaning:")
print(df.isnull().sum())

print(f"\nDataset after cleaning: {df.shape}")

## 4. Feature Engineering

In [ ]:
# Create derived features that represent key productivity factors

# 4.1 Productivity Index: normalized performance relative to study effort
# High productivity = good score with reasonable study time
df['productivity_index'] = (df['exam_score'] / 100) * np.log1p(df['study_hours_per_day'])

# 4.2 Stress Factor: negative impact on productivity
# High social media + Netflix + low sleep + low mental health = high stress
df['stress_factor'] = (
    (df['social_media_hours'] * 0.3 + df['netflix_hours'] * 0.2) / 2 +
    (10 - df['sleep_hours']) * 0.4 +
    (10 - df['mental_health_rating']) * 0.1
)

# 4.3 Engagement Score: positive indicator of study commitment
# Attendance + exercise + extracurricular + study hours
engagement_components = [
    (df['attendance_percentage'] / 100),  # 0-1
    (df['exercise_frequency'] / 6) * 0.5,  # 0-0.5 (if max is ~6 times/week)
    (1.0 if df['extracurricular_participation'] == 'Yes' else 0.0) * 0.3,
    (df['study_hours_per_day'] / 8) * 0.2  # 0-0.2 (if 8 hours is max)
]
df['engagement_score'] = sum(engagement_components) / len(engagement_components)

# 4.4 Time Efficiency: how much output per unit input
# exam_score / study_hours_per_day (avoid division by zero)
df['time_efficiency'] = df['exam_score'] / (df['study_hours_per_day'] + 0.1)

# 4.5 Life Balance Score: inverse of stress combined with positive factors
df['life_balance_score'] = (
    (df['sleep_hours'] / 8) * 0.4 +
    (df['mental_health_rating'] / 10) * 0.3 +
    (df['diet_quality'].map({'Poor': 0.3, 'Fair': 0.6, 'Good': 1.0})) * 0.2 +
    (df['exercise_frequency'] / 6) * 0.1
)

print("Feature Engineering Complete!")
print(f"\nNew derived features:")
print(f"  - productivity_index: {df['productivity_index'].describe()}")
print(f"  - stress_factor: {df['stress_factor'].describe()}")
print(f"  - engagement_score: {df['engagement_score'].describe()}")
print(f"  - time_efficiency: {df['time_efficiency'].describe()}")
print(f"  - life_balance_score: {df['life_balance_score'].describe()}")

## 5. Encode Categorical Variables

In [ ]:
# Encode categorical features
le_dict = {}
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le
    print(f"Encoded {col}: {dict(enumerate(le.classes_))}")

print(f"\nAll categorical features encoded.")
print(f"Dataset shape after encoding: {df.shape}")

## 6. Create Regression Targets

In [ ]:
# TARGET 1: Productivity Score (0-100) - use exam_score directly as baseline
df['productivity_score'] = df['exam_score'].clip(0, 100)

# TARGET 2: Required Study Hours
# Logic: Inverse relationship with productivity
# High performers need fewer hours, low performers need more
# Scale: 2-15 hours per day
base_hours = 8  # baseline
productivity_ratio = df['exam_score'] / 50  # normalize around 50
df['required_hours'] = (base_hours / (productivity_ratio + 0.5)).clip(2, 15)

# TARGET 3: Optimal Break Interval (in minutes)
# Logic: Students under stress need more frequent breaks
# Range: 15-45 minutes
# High mental health + good sleep = longer study intervals (less frequent breaks)
break_base = 30  # baseline 30 min
stress_adjustment = (df['stress_factor'] / 10) * 15  # add up to 15 min if stressed
mental_health_bonus = (df['mental_health_rating'] / 10) * 5  # reduce 0-5 min if mentally healthy
df['optimal_break_interval'] = (break_base + stress_adjustment - mental_health_bonus).clip(15, 45)

print("Regression Targets Created:")
print(f"\n1. Productivity Score (0-100):")
print(df['productivity_score'].describe())
print(f"\n2. Required Study Hours (2-15):")
print(df['required_hours'].describe())
print(f"\n3. Optimal Break Interval (15-45 min):")
print(df['optimal_break_interval'].describe())

## 7. Prepare Features for Modeling

In [ ]:
# Select features (exclude exam_score as it's used in targets)
feature_cols = [
    'age', 'gender', 'study_hours_per_day', 'social_media_hours',
    'netflix_hours', 'part_time_job', 'attendance_percentage', 'sleep_hours',
    'diet_quality', 'exercise_frequency', 'parental_education_level',
    'internet_quality', 'mental_health_rating', 'extracurricular_participation',
    'productivity_index', 'stress_factor', 'engagement_score', 
    'time_efficiency', 'life_balance_score'
]

X = df[feature_cols].copy()
print(f"Features shape: {X.shape}")
print(f"Features: {feature_cols}")

## 8. Train/Test Split

In [ ]:
# Create train/test splits for each target
random_state = 42

# Split for Model 1 & 2 (80/20)
X_train, X_test, y_prod_train, y_prod_test = train_test_split(
    X, df['productivity_score'], test_size=0.2, random_state=random_state
)

_, _, y_hours_train, y_hours_test = train_test_split(
    X, df['required_hours'], test_size=0.2, random_state=random_state
)

_, _, y_break_train, y_break_test = train_test_split(
    X, df['optimal_break_interval'], test_size=0.2, random_state=random_state
)

print(f"Train set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nTrain/Test split: 80/20")

## 9. MODEL 1: Productivity Score Predictor

In [ ]:
print("="*60)
print("TRAINING MODEL 1: Productivity Score Predictor")
print("="*60)
print("Target: Continuous productivity score (0-100)")
print("Algorithm: RandomForestRegressor")
print()

# Create and train model
model1 = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=random_state,
    n_jobs=-1
)

model1.fit(X_train, y_prod_train)
print("✓ Model trained successfully")

# Predictions
y_prod_pred_train = model1.predict(X_train)
y_prod_pred_test = model1.predict(X_test)

# Evaluation metrics
train_mae = mean_absolute_error(y_prod_train, y_prod_pred_train)
test_mae = mean_absolute_error(y_prod_test, y_prod_pred_test)
train_rmse = np.sqrt(mean_squared_error(y_prod_train, y_prod_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_prod_test, y_prod_pred_test))
train_r2 = r2_score(y_prod_train, y_prod_pred_train)
test_r2 = r2_score(y_prod_test, y_prod_pred_test)

print("\nTRAINING METRICS:")
print(f"  MAE:  {train_mae:.4f}")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  R²:   {train_r2:.4f}")

print("\nTEST METRICS:")
print(f"  MAE:  {test_mae:.4f}")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  R²:   {test_r2:.4f}")

# Cross-validation
cv_scores = cross_val_score(model1, X_train, y_prod_train, cv=5, scoring='r2')
print(f"\nCROSS-VALIDATION R² (5-fold): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 10. MODEL 2: Required Study Hours Predictor

In [ ]:
print("="*60)
print("TRAINING MODEL 2: Required Study Hours Predictor")
print("="*60)
print("Target: Daily study hours needed (2-15 hours)")
print("Algorithm: GradientBoostingRegressor")
print()

# Create and train model
model2 = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=random_state
)

model2.fit(X_train, y_hours_train)
print("✓ Model trained successfully")

# Predictions
y_hours_pred_train = model2.predict(X_train)
y_hours_pred_test = model2.predict(X_test)

# Evaluation metrics
train_mae = mean_absolute_error(y_hours_train, y_hours_pred_train)
test_mae = mean_absolute_error(y_hours_test, y_hours_pred_test)
train_rmse = np.sqrt(mean_squared_error(y_hours_train, y_hours_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_hours_test, y_hours_pred_test))
train_r2 = r2_score(y_hours_train, y_hours_pred_train)
test_r2 = r2_score(y_hours_test, y_hours_pred_test)

print("\nTRAINING METRICS:")
print(f"  MAE:  {train_mae:.4f} hours")
print(f"  RMSE: {train_rmse:.4f} hours")
print(f"  R²:   {train_r2:.4f}")

print("\nTEST METRICS:")
print(f"  MAE:  {test_mae:.4f} hours")
print(f"  RMSE: {test_rmse:.4f} hours")
print(f"  R²:   {test_r2:.4f}")

# Cross-validation
cv_scores = cross_val_score(model2, X_train, y_hours_train, cv=5, scoring='r2')
print(f"\nCROSS-VALIDATION R² (5-fold): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 11. MODEL 3: Break Interval Optimizer

In [ ]:
print("="*60)
print("TRAINING MODEL 3: Break Interval Optimizer")
print("="*60)
print("Target: Optimal break interval duration (15-45 minutes)")
print("Algorithm: RandomForestRegressor")
print()

# Create and train model
model3 = RandomForestRegressor(
    n_estimators=150,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=random_state,
    n_jobs=-1
)

model3.fit(X_train, y_break_train)
print("✓ Model trained successfully")

# Predictions
y_break_pred_train = model3.predict(X_train)
y_break_pred_test = model3.predict(X_test)

# Evaluation metrics
train_mae = mean_absolute_error(y_break_train, y_break_pred_train)
test_mae = mean_absolute_error(y_break_test, y_break_pred_test)
train_rmse = np.sqrt(mean_squared_error(y_break_train, y_break_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_break_test, y_break_pred_test))
train_r2 = r2_score(y_break_train, y_break_pred_train)
test_r2 = r2_score(y_break_test, y_break_pred_test)

print("\nTRAINING METRICS:")
print(f"  MAE:  {train_mae:.4f} minutes")
print(f"  RMSE: {train_rmse:.4f} minutes")
print(f"  R²:   {train_r2:.4f}")

print("\nTEST METRICS:")
print(f"  MAE:  {test_mae:.4f} minutes")
print(f"  RMSE: {test_rmse:.4f} minutes")
print(f"  R²:   {test_r2:.4f}")

# Cross-validation
cv_scores = cross_val_score(model3, X_train, y_break_train, cv=5, scoring='r2')
print(f"\nCROSS-VALIDATION R² (5-fold): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 12. Feature Importance Analysis

In [ ]:
# Create figure with 3 subplots for feature importance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Model 1 Feature Importance
importance1 = pd.DataFrame({
    'feature': feature_cols,
    'importance': model1.feature_importances_
}).sort_values('importance', ascending=False).head(10)

axes[0].barh(importance1['feature'], importance1['importance'], color='steelblue')
axes[0].set_title('Model 1: Productivity Score\nTop 10 Features', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importance')

# Model 2 Feature Importance
importance2 = pd.DataFrame({
    'feature': feature_cols,
    'importance': model2.feature_importances_
}).sort_values('importance', ascending=False).head(10)

axes[1].barh(importance2['feature'], importance2['importance'], color='coral')
axes[1].set_title('Model 2: Required Study Hours\nTop 10 Features', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance')

# Model 3 Feature Importance
importance3 = pd.DataFrame({
    'feature': feature_cols,
    'importance': model3.feature_importances_
}).sort_values('importance', ascending=False).head(10)

axes[2].barh(importance3['feature'], importance3['importance'], color='mediumseagreen')
axes[2].set_title('Model 3: Break Interval Optimizer\nTop 10 Features', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.show()

print("Feature importance analysis complete!")

## 13. Residual Analysis

In [ ]:
# Residual plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Model 1 Residuals
residuals1 = y_prod_test - y_prod_pred_test
axes[0, 0].scatter(y_prod_pred_test, residuals1, alpha=0.6, color='steelblue')
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_title('Model 1: Residuals vs Predictions', fontweight='bold')
axes[0, 0].set_xlabel('Predicted Productivity Score')
axes[0, 0].set_ylabel('Residuals')

axes[1, 0].hist(residuals1, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Model 1: Distribution of Residuals', fontweight='bold')
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')

# Model 2 Residuals
residuals2 = y_hours_test - y_hours_pred_test
axes[0, 1].scatter(y_hours_pred_test, residuals2, alpha=0.6, color='coral')
axes[0, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 1].set_title('Model 2: Residuals vs Predictions', fontweight='bold')
axes[0, 1].set_xlabel('Predicted Study Hours')
axes[0, 1].set_ylabel('Residuals')

axes[1, 1].hist(residuals2, bins=30, color='coral', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Model 2: Distribution of Residuals', fontweight='bold')
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frequency')

# Model 3 Residuals
residuals3 = y_break_test - y_break_pred_test
axes[0, 2].scatter(y_break_pred_test, residuals3, alpha=0.6, color='mediumseagreen')
axes[0, 2].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 2].set_title('Model 3: Residuals vs Predictions', fontweight='bold')
axes[0, 2].set_xlabel('Predicted Break Interval (min)')
axes[0, 2].set_ylabel('Residuals')

axes[1, 2].hist(residuals3, bins=30, color='mediumseagreen', alpha=0.7, edgecolor='black')
axes[1, 2].set_title('Model 3: Distribution of Residuals', fontweight='bold')
axes[1, 2].set_xlabel('Residuals')
axes[1, 2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print("Residual analysis complete!")

## 14. Export Models & Metadata

In [ ]:
# Create artifacts directory
os.makedirs('artifacts', exist_ok=True)

# Save models
model1_path = 'artifacts/productivity_score_model.pkl'
model2_path = 'artifacts/required_hours_model.pkl'
model3_path = 'artifacts/break_interval_model.pkl'

joblib.dump(model1, model1_path)
joblib.dump(model2, model2_path)
joblib.dump(model3, model3_path)

print("✓ Models saved:")
print(f"  - {model1_path}")
print(f"  - {model2_path}")
print(f"  - {model3_path}")

# Create comprehensive metadata
metadata = {
    "project": "AI Productivity Maximizer",
    "version": "2.0",
    "created": datetime.now().isoformat(),
    "models": {
        "productivity_score": {
            "description": "Predicts student productivity score (0-100)",
            "algorithm": "RandomForestRegressor",
            "hyperparameters": {
                "n_estimators": 200,
                "max_depth": 12,
                "min_samples_split": 5,
                "min_samples_leaf": 2
            },
            "test_r2": float(test_r2),
            "test_mae": float(mean_absolute_error(y_prod_test, y_prod_pred_test)),
            "top_features": importance1.head(5).to_dict('records')
        },
        "required_hours": {
            "description": "Predicts required daily study hours (2-15)",
            "algorithm": "GradientBoostingRegressor",
            "hyperparameters": {
                "n_estimators": 200,
                "learning_rate": 0.1,
                "max_depth": 5,
                "min_samples_split": 5,
                "min_samples_leaf": 2
            },
            "test_r2": float(r2_score(y_hours_test, y_hours_pred_test)),
            "test_mae": float(mean_absolute_error(y_hours_test, y_hours_pred_test)),
            "top_features": importance2.head(5).to_dict('records')
        },
        "break_interval": {
            "description": "Predicts optimal break interval (15-45 min)",
            "algorithm": "RandomForestRegressor",
            "hyperparameters": {
                "n_estimators": 150,
                "max_depth": 10,
                "min_samples_split": 5,
                "min_samples_leaf": 2
            },
            "test_r2": float(r2_score(y_break_test, y_break_pred_test)),
            "test_mae": float(mean_absolute_error(y_break_test, y_break_pred_test)),
            "top_features": importance3.head(5).to_dict('records')
        }
    },
    "features": feature_cols,
    "feature_ranges": {
        col: {"min": float(X[col].min()), "max": float(X[col].max())}
        for col in feature_cols
    },
    "training_data": {
        "total_samples": len(df),
        "train_samples": len(X_train),
        "test_samples": len(X_test)
    }
}

# Save metadata
metadata_path = 'artifacts/model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ Metadata saved: {metadata_path}")

## 15. Example Predictions

In [ ]:
# Create example predictions for different student profiles
print("="*70)
print("EXAMPLE PREDICTIONS FOR DIFFERENT STUDENT PROFILES")
print("="*70)

# Get 5 diverse student examples from test set
indices = [0, len(X_test)//4, len(X_test)//2, 3*len(X_test)//4, -1]

for i, idx in enumerate(indices):
    print(f"\nStudent Profile {i+1}:")
    print("-" * 50)
    
    student_features = X_test.iloc[idx:idx+1]
    
    # Get predictions
    prod_score = model1.predict(student_features)[0]
    study_hours = model2.predict(student_features)[0]
    break_interval = model3.predict(student_features)[0]
    
    # Display features
    print(f"  Study hours/day: {student_features['study_hours_per_day'].values[0]:.1f}")
    print(f"  Sleep hours: {student_features['sleep_hours'].values[0]:.1f}")
    print(f"  Mental health: {student_features['mental_health_rating'].values[0]:.0f}/10")
    print(f"  Attendance: {student_features['attendance_percentage'].values[0]:.1f}%")
    
    print(f"\n  PREDICTIONS:")
    print(f"  ✓ Productivity Score: {prod_score:.1f}/100")
    print(f"  ✓ Recommended Study Time: {study_hours:.1f} hours/day")
    print(f"  ✓ Optimal Break Interval: {break_interval:.0f} minutes")
    
    # Interpretation
    if prod_score > 75:
        print(f"  → High performer: Maintain current routine")
    elif prod_score > 50:
        print(f"  → Average performer: Can improve with focused study")
    else:
        print(f"  → Needs support: Increase study hours and reduce distractions")

print("\n" + "="*70)
print("Regression model training and evaluation complete!")
print("="*70)